# The GAMLSS framework

**GAMLSS** (Generalized Additive Models for Location, Scale and Shape) models
multiple parameters of a response distribution as additive functions of
covariates. This chapter defines those predictors, derives likelihood scores
and working weights, and describes the RS fitting algorithm.
[](gamlss.ipynb) describes how `gamboostLSS` fits the same model class by boosting.

```{admonition} Prerequisites
:class: tip
This chapter assumes [](prerequisites.ipynb) (exponential family, GLMs, IRLS) and
[](from_glm_to_boosting.ipynb) §5 (the motivation for distributional regression).
```



## 1. From GLM to GAMLSS



### 1.1 GLMs and GAMs

A **GLM** ([]{}`prerequisites.md` §4) models the conditional mean
$\mu = \mathbb{E}[Y \mid X]$ as a linear function of covariates through a link:
$g(\mu) = \eta = X\beta$. A **GAM** replaces the linear predictor with smooth
functions: $g(\mu) = \beta_0 + f_1(x_1) + \dots + f_p(x_p)$. In both cases,
**only the mean is modelled** — the variance, skewness, and other distributional
properties are assumed constant or determined by the mean (e.g., Poisson:
$\operatorname{Var} = \mu$).

```{figure} ../figures/mermaid-b079d6fea545.svg
:name: mermaid-b079d6fea545
:alt: Conditional response distribution with a modeled mean predictor, constant dispersion, and no separate model for shape or skewness.

A conventional GLM or GAM relates covariates to the conditional mean while treating dispersion as fixed or as determined by that mean.
```



### 1.2 GAMLSS

**GAMLSS** {cite}`rigby2005generalized,stasinopoulos2007generalized` relaxes
this: **every** parameter of the conditional distribution is modelled as a
function of covariates. If the response distribution has $K$ parameters
$\theta_1, \dots, \theta_K$ (location, scale, shape, …), each gets its own
additive predictor:

$$
g_k(\theta_k) = \eta_k = \beta_{k,0} + f_{k,1}(x_1) + \dots + f_{k,p}(x_p), \qquad k = 1,\dots,K.
$$

```{figure} ../figures/mermaid-932dc08737ca.svg
:name: mermaid-932dc08737ca
:alt: A response distribution whose location, scale, and shape parameters each have their own covariate-dependent additive predictor.

GAMLSS assigns a separate additive predictor and link function to every modeled distribution parameter.
```



### 1.3 Modelling more than the mean

Modelling only the mean misses structure that is scientifically important:

```{figure} ../figures/fig-heteroscedastic.svg
:label: fig-heteroscedastic
:alt: Two scenarios where mean-only models are insufficient — heteroscedastic data (spread grows with x) and treatment groups whose distributions differ in shape (symmetric vs right-skewed)

Two data-generating patterns a mean-only model cannot represent: spread that grows with the covariate, and treatment groups that share a mean but differ in shape.
```

- **Income data**: mean income rises with education, but so does *inequality*
  (the spread). A GAMLSS model captures both $\mu(\text{edu})$ and
  $\sigma(\text{edu})$.
- **Survival analysis**: a treatment may extend median survival *and* reduce
  variability in survival times. Model both $\mu(\text{treatment})$ and
  $\sigma(\text{treatment})$.
- **Test scores**: item difficulty affects the average score and the
  skewness (hard items produce left-skewed distributions).
- **Financial returns**: volatility clustering means $\sigma$ (scale) varies
  over time — the entire GARCH literature is about modelling $\sigma(t)$.



## 2. The GAMLSS model formally



### 2.1 The four components

A GAMLSS model has four components, generalizing the three of a GLM/GAM:

1. **The response distribution** $d(y \mid \boldsymbol\theta)$, parameterised by
   a $K$-vector $\boldsymbol\theta = (\theta_1, \dots, \theta_K)$ of distribution
   parameters. These can be location, scale, shape, skewness, kurtosis, or
   mixing parameters — *any* property of the distribution.
2. **The monotonic link functions** $g_k$, one per parameter, mapping
   $\theta_k$ to the real line (or appropriate domain): $\eta_k = g_k(\theta_k)$.
3. **The additive predictors** $\eta_k$, each a sum of (possibly smooth)
   effects of covariates.
4. **The fitting algorithm** — either classical (RS / backfitting, §3) or
   boosting (next chapter, [](gamlss.ipynb)).



### 2.2 The distribution parameter naming convention

GAMLSS uses a consistent naming convention for the $K$ parameters, reflecting
their role in the distribution:

| Parameter | Symbol | Role | Typical link |
|---|---|---|---|
| Location | $\mu$ | Mean / median / mode | identity or log |
| Scale | $\sigma$ | Standard deviation / spread | log |
| Shape 1 | $\nu$ | Skewness | log or logit |
| Shape 2 | $\tau$ | Kurtosis / tail weight | log |

For a **Gaussian**, $K = 2$: $(\mu, \sigma)$. For a **Student-t**, $K = 3$:
$(\mu, \sigma, \nu)$ where $\nu$ is the degrees of freedom (tail weight). For
**zero-inflated negative binomial**, $K = 3$: $(\mu, \sigma, \nu)$ where $\mu$
is the mean, $\sigma$ is the dispersion, and $\nu$ is the zero-inflation
probability. The `mboostrs` family catalogue is in [](gamlss.ipynb).



### 2.3 The link functions

Each parameter $\theta_k$ has a link $g_k$ that maps it from its natural domain
to the real line (the additive predictor scale):

| Parameter domain | Link $g_k$ | Inverse $g_k^{-1}$ | Example |
|---|---|---|---|
| $\mathbb{R}$ (location) | identity | $\theta = \eta$ | Gaussian $\mu$ |
| $(0, \infty)$ (scale, shape) | $\log$ | $\theta = e^\eta$ | Gaussian $\sigma$, Gamma $\sigma$ |
| $(0, 1)$ (probability) | $\operatorname{logit}$ | $\theta = (1+e^{-\eta})^{-1}$ | ZIPo $\pi$, Beta $\mu$ |
| $(0, \infty)$ (precision) | $\log$ | $\theta = e^\eta$ | Beta $\phi$ |

The link ensures the parameter stays in its valid domain regardless of the
predictor value. Choosing the right link matters: a $\log$ link for $\sigma$
guarantees positivity; a logit link for a probability guarantees it stays in
$(0,1)$.



## 3. Fitting GAMLSS: the RS algorithm



### 3.1 IRLS for a single parameter

A single-parameter GLM is fit by IRLS ([]{}`prerequisites.md` §4.2): iterate
weighted least squares until convergence. GAMLSS has $K$ coupled parameters, so
IRLS must be generalized to handle the **joint** optimization. The standard
approach is the **RS algorithm** {cite}`rigby2005generalized`, a
Gauss-Seidel-type (cyclic) generalization of IRLS.



### 3.2 The RS algorithm

The RS (Rigby–Stasinopoulos) algorithm cycles through the $K$ parameters,
updating each by a penalized IRLS step while holding the others fixed.

```{admonition} Algorithm — RS (cyclic penalized IRLS for GAMLSS)
:class: tip

**Input:** distribution $d(y \mid \boldsymbol\theta)$, links $g_1, \dots, g_K$,
design/penalty matrices $(X_k, K_k)$ for each parameter, tolerance $\varepsilon$.

1. **Initialize** $\hat\eta_k^{[0]}$ for each parameter (e.g., method-of-moments
   or default values).
2. **Repeat** for cycles $t = 1, 2, \dots$:
   **For** $k = 1, \dots, K$ (sequentially):
   a. Compute the **partial derivative** of the log-likelihood w.r.t. $\eta_k$:
      $s_k^{(i)} = \partial \log d / \partial \eta_k^{(i)}$ (the score for
      parameter $k$ at observation $i$).
   b. Compute the **working weights** $W_k^{(i)}$ — the expected or observed
      Fisher information for $\eta_k$, evaluated at current parameters.
   c. Form the **working response** $z_k^{(i)} = \hat\eta_k^{(i)} + W_k^{-1}\, s_k^{(i)}$.
   d. Solve the **penalized weighted least squares**:
      $\hat\beta_k^{\text{new}} = (X_k^\top W_k X_k + \lambda_k K_k)^{-1} X_k^\top W_k z_k$.
   e. Update $\hat\eta_k \gets X_k \hat\beta_k^{\text{new}}$.
   **End for**.
3. **Until** the change in the global deviance
   $D = -2\sum_i \log d(y_i \mid \hat{\boldsymbol\theta}_i)$ is below
   $\varepsilon$.
```

```{figure} ../figures/mermaid-15fd1522d4b3.svg
:name: mermaid-15fd1522d4b3
:alt: Cyclic GAMLSS updates for location, scale, and shape followed by a global-deviance convergence check.

GAMLSS backfitting updates distribution parameters sequentially with the latest values of the other parameters and stops when deviance stabilizes.
```

Each parameter update uses the **latest** values of the earlier parameters
(Gauss-Seidel), not the values from the start of the cycle (Jacobi).



### 3.3 The backfitting inner loop

Step 2d (penalized weighted least squares for parameter $k$) is itself solved
by **backfitting** when the predictor $\eta_k$ is a sum of multiple smooth
effects (a GAM for parameter $k$). Backfitting cycles through the smooth terms,
fitting each to the partial residuals while holding the others fixed:

```
Backfitting for parameter k (inner loop of RS step 2d):

  η_k = f_{k,1}(x₁) + f_{k,2}(x₂) + ... + f_{k,p}(x_p)

  Repeat until convergence:
    For each effect j = 1, ..., p:
      partial residual: r_j = z_k − η_k + f_{k,j}(x_j)
                        (working response minus all OTHER effects)
      update:  f_{k,j} $\leftarrow$ S_j · r_j    (smooth fit via penalized LS)
      update:  η_k $\leftarrow$ η_k − old_f + new_f
```

This two-level structure (RS outer loop cycling over parameters, backfitting
inner loop cycling over effects within each parameter) is the classical GAMLSS
fitting algorithm. It is implemented in the R `gamlss` package.



### 3.4 Convergence of the RS algorithm

The RS algorithm converges under mild conditions: the log-likelihood must be
concave in each parameter (holding others fixed), and the penalties must be
positive-semidefinite. These conditions must be checked for the chosen family
and predictor parameterization; distributional likelihoods need not be jointly
concave. Convergence is typically reached in 5–20 cycles for well-conditioned
problems, but the iteration count alone does not establish a global optimum.

```{admonition} RS and boosting
:class: important

The **RS algorithm** solves the GAMLSS model to convergence — it finds the
penalized MLE. The smoothing parameters $\lambda_k$ must be chosen *before*
fitting (e.g., by GAIC or cross-validation over a grid).

**GAMLSS boosting** (next chapter) does NOT solve to convergence — it takes
small incremental steps and stops early. The regularization comes from the
number of boosting iterations ($m_{\text{stop}}$), not from a pre-specified
$\lambda$. This means:

- Boosting does **automatic variable selection** (irrelevant effects stay at
  zero); RS with a fixed $\lambda$ does not (all effects are non-zero unless
  the penalty is infinite).
- Non-cyclic boosting tunes one $m_{\text{stop}}$; cyclic boosting tunes a
  vector of per-parameter stopping iterations.
- RS repeatedly solves conditional penalized likelihood problems. A global
  optimum requires appropriate convexity and convergence conditions.
  Boosting builds a path through greedy learner updates.

Both produce valid GAMLSS fits; they are complementary tools. `mboostrs`
implements the boosting version; the R `gamlss` package implements RS.
```



## 4. The likelihood, score, and information for GAMLSS



### 4.1 The log-likelihood

The GAMLSS log-likelihood is

$$
\ell(\boldsymbol\eta) = \sum_{i=1}^n \log d\!\big(y_i \mid g_1^{-1}(\eta_{1,i}), \dots, g_K^{-1}(\eta_{K,i})\big).
$$

The global deviance is $D = -2\ell$. Maximizing $\ell$ (minimizing $D$) over the
additive predictors $\eta_1, \dots, \eta_K$ gives the GAMLSS fit.



### 4.2 The score (partial derivative)

The **score** for parameter $k$ is the partial derivative of $\ell$ with respect
to $\eta_k$. By the chain rule (through the link $g_k$):

$$
s_k^{(i)} = \frac{\partial \log d}{\partial \eta_k^{(i)}}
          = \frac{\partial \log d}{\partial \theta_k^{(i)}} \cdot
            \frac{\partial \theta_k^{(i)}}{\partial \eta_k^{(i)}}
          = \frac{\partial \log d}{\partial \theta_k} \cdot \frac{1}{g_k'(\theta_k^{(i)})}.
$$

The first factor is the score in the natural parameterisation; the second
factor converts from $\theta_k$-scale to $\eta_k$-scale via the link derivative.

For a **log link** ($g_k(\theta) = \log\theta$), $g_k'(\theta) = 1/\theta$, so
$\partial\theta/\partial\eta = \theta = e^{\eta_k}$. The chain-rule factor
**rescales** the score by the current parameter value.

```{figure} ../figures/mermaid-a0db614dc3de.svg
:name: mermaid-a0db614dc3de
:alt: Relationship among predictor scale, parameter scale, the natural-parameter likelihood score, and the score rescaled by the link derivative.

The chain rule converts a likelihood score on the distribution-parameter scale to the additive-predictor scale through the inverse-link derivative.
```

The link derivative rescales the natural-parameter score to the predictor
scale:

| Link | $g'_k(\theta)$ | Score on predictor scale |
|---|---|---|
| $\log$ | $1/\theta$ | $s_k = \theta_k \cdot \partial\log d/\partial\theta_k = e^{\eta_k} \cdot \text{score}$ |
| identity | $1$ | $s_k = \partial\log d/\partial\theta_k$ (no rescaling) |
| $\operatorname{logit}$ | $1/(\theta(1-\theta))$ | $s_k = \theta(1-\theta) \cdot \partial\log d/\partial\theta_k$ |



### 4.3 The Fisher information (working weights)

The **working weights** for parameter $k$ are the (negative) second derivative
of the log-likelihood with respect to $\eta_k$ — the Fisher information for the
predictor:

$$
W_k^{(i)} = -\mathbb{E}\!\Big[\frac{\partial^2 \log d}{(\partial \eta_k^{(i)})^2}\Big].
$$

This is the quantity that determines the precision of the parameter estimate:
large $W_k$ means the data is informative about $\eta_k$ (steep likelihood);
small $W_k$ means it is not (flat likelihood). The working weights enter the
penalized IRLS step as the diagonal weight matrix.



### 4.4 Worked example: GaussianLSS score and weights

For the Gaussian distribution $d(y \mid \mu, \sigma)$ with identity link for
$\mu$ and log link for $\sigma$:

$$
\log d = -\frac12\log(2\pi) - \log\sigma - \frac{(y-\mu)^2}{2\sigma^2}.
$$

**Score for $\mu$** (identity link, $\partial\mu/\partial\eta_\mu = 1$):

$$
s_\mu^{(i)} = \frac{\partial \log d}{\partial \mu^{(i)}} = \frac{y_i - \mu_i}{\sigma_i^2}.
$$

This is the **precision-weighted residual** — the usual regression score, scaled
by $1/\sigma^2$. When $\sigma$ is large (noisy), the score is small (less
information about $\mu$).

**Score for $\sigma$** (log link, $\partial\sigma/\partial\eta_\sigma = \sigma$):

$$
s_\sigma^{(i)} = \frac{\partial \log d}{\partial \sigma^{(i)}} \cdot \sigma_i
                = \Big(-\frac{1}{\sigma_i} + \frac{(y_i - \mu_i)^2}{\sigma_i^3}\Big)\sigma_i
                = -1 + \frac{(y_i - \mu_i)^2}{\sigma_i^2}.
$$

This is $-1 + r_i^2/\sigma_i^2$ where $r_i = y_i - \mu_i$. When the squared
residual **exceeds** the expected value $\sigma^2$, the score is positive
(drive $\sigma$ up); when it is below, the score is negative (drive $\sigma$
down). This is the dispersion-detection mechanism.

```{figure} ../figures/fig-gaussian-scores.svg
:label: fig-gaussian-scores
:alt: GaussianLSS μ-score (precision-weighted residual, zero at y=μ) and σ-score (−1 + (y−μ)²/σ², zero at |y−μ|=σ) as functions of the standardized residual

The two GaussianLSS scores against the standardized residual. The $\mu$-score is a precision-weighted residual, zero at $y = \mu$; the $\sigma$-score is zero when the squared residual equals $\sigma^2$, positive above it and negative below, which is what drives the scale fit.
```

**Working weight for $\mu$:**

$$
W_\mu^{(i)} = -\mathbb{E}\!\Big[\frac{\partial^2 \log d}{\partial \mu^2}\Big] = \frac{1}{\sigma_i^2}.
$$

**Working weight for $\sigma$:**

$$
W_\sigma^{(i)} = -\mathbb{E}\!\Big[\frac{\partial^2 \log d}{(\partial\eta_\sigma)^2}\Big] = 2.
$$

(The second derivative of $-\log\sigma - (y-\mu)^2/(2\sigma^2)$ w.r.t.
$\eta_\sigma = \log\sigma$ evaluates to a constant under the expectation.)
These weights appear in the penalized IRLS step for each parameter.



## 5. Choosing smoothing parameters: GAIC and cross-validation



### 5.1 Smoothing parameters in GAMLSS

The GAMLSS model has $K$ smoothing parameters $\lambda_1, \dots, \lambda_K$ (one
per parameter), each controlling the smoothness of its additive predictor. In
the RS algorithm, these must be chosen *before* or *during* fitting. There are
two approaches:



### 5.2 Generalized AIC (GAIC)

The **GAIC** criterion generalizes AIC to multiple smoothing parameters:

$$
\mathrm{GAIC}_\#(\lambda_1, \dots, \lambda_K) = -2\ell(\hat{\boldsymbol\eta}) + \#\,\sum_{k=1}^K \mathrm{df}_k(\lambda_k),
$$

where $\#$ is a penalty multiplier:
- $\# = 2$ gives **AIC** (liberal, tends to overfit).
- $\# = \log n$ gives **BIC** (conservative, tends to underfit).
- $\# = 2.5$ or $3$ are intermediate choices used in practice.

The optimal $(\lambda_1, \dots, \lambda_K)$ is found by minimizing GAIC over a
grid (or by a numerical optimizer). This is computationally expensive: each
evaluation requires a full RS fit.



### 5.3 Cross-validation

K-fold cross-validation evaluates the out-of-sample negative log-likelihood:

$$
\mathrm{CV}(\lambda_1, \dots, \lambda_K) = \frac{1}{K}\sum_{k=1}^{K} \sum_{i \in \text{fold}_k} -\log d(y_i \mid \hat{\boldsymbol\theta}^{(-k)}),
$$

where $\hat{\boldsymbol\theta}^{(-k)}$ is the fit with fold $k$ held out. CV is
more expensive than GAIC (requires $K$ full fits), and evaluates predictive
performance on rows held out from each fit.



### 5.4 Tuning GAMLSS by boosting

The classical GAMLSS tuning problem is $K$-dimensional (one $\lambda_k$ per
parameter, each chosen by GAIC or CV over a grid). This becomes intractable for
$K \ge 3$.

**GAMLSS boosting** replaces the $K$ smoothing parameters with a single
$m_{\text{stop}}$ (non-cyclic) or a $K$-vector of $m_{\text{stop},k}$ (cyclic).
Early stopping limits the cumulative updates to all effects. Base learners
retain their own fixed penalties; the stopping iteration controls the
complexity accumulated across repeated updates.

```{figure} ../figures/fig-tuning-space.svg
:label: fig-tuning-space
:alt: Two-panel comparison — RS algorithm requires a 3-D grid of smoothing parameters (λ₁, λ₂, λ₃), while non-cyclic GAMLSS boosting needs only a 1-D m_stop CV curve

What has to be tuned. The RS algorithm searches a smoothing parameter per distribution parameter, a $K$-dimensional grid; non-cyclic boosting searches one $m_{\text{stop}}$ along a single cross-validation curve.
```

For $K=3$ parameters with 20 candidate values each, the RS grid search
requires $20^3 = 8000$ full fits. Non-cyclic boosting instead tunes a single
stopping iteration along one path per fold. Cyclic boosting retains a joint
$K$-dimensional stopping grid: equal-budget points share checkpoints, while
unequal-budget points generally require separate fits.



## 6. Diagnostics for GAMLSS



### 6.1 The (normalized quantile) residuals

GAMLSS uses **normalized randomized quantile residuals** (Dunn–Smyth residuals {cite}`dunn1996randomized`)
for diagnostics. For a continuous response, the residual is

$$
r_i = \Phi^{-1}\!\big(D(y_i \mid \hat{\boldsymbol\theta}_i)\big),
$$

where $D$ is the fitted CDF and $\Phi^{-1}$ is the standard normal quantile
function. If the model is correct, the $r_i$ should be approximately
$N(0,1)$. For discrete responses, randomization is needed to break ties.

These residuals are checked for:
- **Normality** (Q-Q plot): should lie on the diagonal.
- **Homoscedasticity** (residuals vs fitted): no pattern.
- **Trends** (residuals vs each covariate): no systematic structure.

```{figure} ../figures/fig-qq-residuals.svg
:label: fig-qq-residuals
:alt: Q-Q plot of normalized quantile residuals — good fit (points lie on the diagonal, consistent with N(0,1)) vs bad fit (S-shape indicating a distribution-family mismatch such as skewness)

Normalized quantile residuals against normal quantiles. A well-specified family puts the points on the diagonal; the S-shape on the right is the signature of a family whose skewness does not match the data.
```

If the Q-Q plot deviates systematically, the distribution family is wrong.
Try a different family (e.g., Student-t instead of Gaussian for heavy tails).



### 6.2 Distribution selection

Choosing the right distribution shapes every later modeling decision in
GAMLSS. A practical workflow:

1. Start with the simplest family (GaussianLSS for continuous, Poisson for
   counts).
2. Examine the residuals: heavy tails $\rightarrow$ Student-t; skewness $\rightarrow$ Gamma or
   log-Normal; excess zeros $\rightarrow$ zero-inflated.
3. Compare families by GAIC or CV: the family with the lowest criterion is
   preferred.
4. Check that the estimated shape parameters make sense (e.g., Student-t $\nu$
   should be finite; if $\hat\nu > 30$, the Gaussian is adequate).



## 7. GAMLSS families in `mboostrs`

The `mboostrs` GAMLSS family catalogue (see [](gamlss.ipynb) for the full table)
covers the most common distributions:

| Family | $K$ | Parameters | Use case |
|---|---|---|---|
| `GaussianLSS` | 2 | $\mu, \sigma$ | Heteroscedastic continuous |
| `GammaLSS` | 2 | $\mu, \sigma$ | Positive right-skewed |
| `StudentTLSS` | 3 | $\mu, \sigma, \nu$ | Heavy-tailed continuous |
| `BetaLSS` | 2 | $\mu, \phi$ | Proportions in $(0,1)$ |
| `NBinomialLSS` | 2 | $\mu, \sigma$ | Overdispersed counts |
| `LogNormalLSS` | 2 | $\mu, \sigma$ | Log-normal positive |
| `ZIPoLSS` | 2 | $\mu, \sigma$ | Zero-inflated counts |
| `ZINBLSS` | 3 | $\mu, \sigma, \nu$ | Zero-inflated overdispersed counts |
| `DirichletLSS` | $K$ | $\alpha_1, \dots, \alpha_K$ | Compositional data |

Each family's density, score functions, and link choices are derived in
[](gamlss.ipynb). The framework remains the same across families; the density
$d(y \mid \boldsymbol\theta)$ and its derivatives vary.



## 8. The connection to boosting

This chapter has developed GAMLSS as a **model class**: a flexible family of
distributions with covariate-dependent parameters. The **fitting algorithm**
(RS / backfitting) is one way to estimate the model; **boosting** is another.

The next chapter, [](gamlss.ipynb), shows how model-based boosting fits GAMLSS
models. The key ideas carry over directly:

- The **loss** is the joint negative log-likelihood (same as §4.1).
- The **negative gradient** for parameter $k$ is the **score** $s_k$ (same as
  §4.2) — this is the "pseudo-residual" that boosting fits.
- The **base learners** replace the smooth terms $f_{k,j}$ — each parameter has
  its own set, just as in the RS algorithm.
- The **regularization** shifts from smoothing parameters $\lambda_k$ (RS) to
  boosting iterations $m_{\text{stop}}$ (boosting) — the simplification
  described in §5.4.



## Summary

- GAMLSS extends GLMs/GAMs by modelling **all** parameters of a response
  distribution (mean, scale, shape) as additive functions of covariates,
  using an appropriate link for each parameter.
- The RS (Rigby–Stasinopoulos) algorithm is a backfitting procedure that
  cycles through parameters, each step using a penalised likelihood fit.
  Boosting replaces the backfitting step with a single gradient-descent
  iteration per parameter, and $m_{\text{stop}}$ replaces $\lambda$ as the
  regularizer.
- The negative gradient in boosting is the **score**
  $s_k = \partial \log d(y \mid \boldsymbol\theta)/\partial \eta_k$, the
  same quantity that appears in the Fisher-scoring step of the RS algorithm.



## See also

- [](../usage/gamboostlss) — runnable examples of GAMLSS boosting with the
  GaussianLSS, StudentTLSS, and other families.
- [](../implementation/lss_engine) — the Rust LSS engine: cyclic vs non-cyclic
  updates, the per-parameter Gram solver pool, and the `LssPath` structure.

```{bibliography}
:filter: docname in docnames
```